[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/main/notebooks/case_studies/data_analysis_assistant/langgraph.ipynb)

# Data-analysis assistant — LangGraph

**Task.** Answer a user-specified descriptive analysis question using a typed allowlisted operation.

**Conceptual stages.** `inspect_data → select_analysis → validate_request → execute_analysis → validate_result → report`.

The model selects an operation and writes the final explanation; deterministic Python validates columns, performs the calculation and checks the versioned oracle.


In [ ]:
import os

# 1. Choose the model provider; then use Run All. No terminal command is needed.
MODEL_PROVIDER = "mock"  # mock | local | api
MOCK_MODEL_NAME = "analysis-case-v1"
LOCAL_MODEL_NAME = "Qwen3-0.6B-Q8_0"
LOCAL_MODEL_PATH = os.getenv(
    "AGENTIC_TUTORIAL_LOCAL_MODEL_PATH",
    "models/local/Qwen3-0.6B-Q8_0.gguf",
)
API_MODEL_NAME = "gemini-3.1-flash-lite"
SAVE_API_CREDENTIAL = True  # saves hidden input to ignored .private/ storage
# Mock runtime is under one minute on CPU; local and API runs can be slower.

ANALYSIS_QUESTION = "Which intervention has the largest mean reduction in household waste?"

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "langgraph==1.2.9", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import csv
import json
from pathlib import Path
from typing import ClassVar, Literal

from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, ConfigDict

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import GenerationSettings, create_model  # noqa: E402
from agentic_tutorial.notebook import prepare_gemini_api_key  # noqa: E402
from agentic_tutorial.schemas import Message, MessageRole  # noqa: E402
from agentic_tutorial.tools import AnalysisRequest, file_sha256, summarise_reduction  # noqa: E402

data_path = ROOT / "data/data_analysis_assistant/household_waste.csv"
expected = json.loads((ROOT / "data/data_analysis_assistant/expected_summary.json").read_text())
fixture_path = ROOT / "data/data_analysis_assistant/case_mock.json"
fixture = json.loads(fixture_path.read_text())
if MODEL_PROVIDER == "api":
    prepare_gemini_api_key(ROOT, save=SAVE_API_CREDENTIAL)

## Matched workflow

The six conceptual stages match the paper. Framework-native abstractions express control, while deterministic Python retains execution, validation and provenance boundaries.


In [ ]:
# --- Matched case-study implementation ---


class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class AnalysisDecision(Strict):
    schema_id: ClassVar[str] = "analysis.decision.v1"
    tool: Literal["mean_reduction_by_intervention"]
    group_column: Literal["intervention"]
    before_column: Literal["before_kg"]
    after_column: Literal["after_kg"]


class Report(Strict):
    schema_id: ClassVar[str] = "analysis.report.v1"
    report: str
    groups: tuple[str, ...]


AnalysisDecision.model_rebuild(_types_namespace={"Literal": Literal})


def create_client():
    model_names = {"mock": MOCK_MODEL_NAME, "local": LOCAL_MODEL_NAME, "api": API_MODEL_NAME}
    model_path = ROOT / LOCAL_MODEL_PATH if MODEL_PROVIDER == "local" else None
    return create_model(
        provider=MODEL_PROVIDER,
        mock_fixture_path=str(fixture_path),
        model=model_names[MODEL_PROVIDER],
        model_path=model_path,
        metadata_path=ROOT / "models/local/model_metadata.json",
        settings=GenerationSettings(temperature=0.0, max_output_tokens=1024, seed=0),
        options={"timeout_seconds": 180.0},
    )


def user(text):
    return Message(role=MessageRole.USER, content=text)


async def propose(client, schema, text):
    response = await client.generate([user(text)], response_schema=schema)
    return schema.model_validate(response.structured_output)


def inspect_data():
    with data_path.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    if not rows:
        raise ValueError("The dataset is empty.")
    return {
        "rows": len(rows),
        "columns": tuple(rows[0]),
        "dataset_sha256": file_sha256(data_path),
    }


async def select_analysis(client, question, inspection):
    return await propose(
        client,
        AnalysisDecision,
        (
            f"Question: {question!r}. Available columns: {inspection['columns']}. "
            "Select one permitted analysis operation and valid columns. "
            "Do not write code or invent numeric results."
        ),
    )


def validate_request(decision, inspection):
    allowed_columns = set(inspection["columns"])
    requested_columns = {
        decision.group_column,
        decision.before_column,
        decision.after_column,
    }
    return decision.tool == "mean_reduction_by_intervention" and requested_columns.issubset(
        allowed_columns
    )


def execute_analysis(decision):
    request = AnalysisRequest(
        group_column=decision.group_column,
        before_column=decision.before_column,
        after_column=decision.after_column,
    )
    return summarise_reduction(data_path, request)


def validate_result(result):
    return result == expected["mean_reduction_kg"]


async def report(client, question, result):
    return await propose(
        client,
        Report,
        (
            f"Answer {question!r} using only this validated result: {result}. "
            "Include every result key exactly once in groups and do not infer causality."
        ),
    )


def validate_report(report_output, result):
    return set(report_output.groups) == set(result)


def build_graph(question=ANALYSIS_QUESTION):
    client = create_client()

    def inspect_data_node(state):
        inspection = inspect_data()
        return {
            **state,
            "question": question,
            "inspection": inspection,
            "trace": [{"event": "inspect_data", **inspection}],
            "model_calls": 0,
        }

    async def select_analysis_node(state):
        decision = await select_analysis(client, state["question"], state["inspection"])
        return {
            **state,
            "decision": decision,
            "model_calls": 1,
            "trace": [*state["trace"], {"event": "select_analysis", "tool": decision.tool}],
        }

    def validate_request_node(state):
        valid = validate_request(state["decision"], state["inspection"])
        return {
            **state,
            "request_valid": valid,
            "trace": [*state["trace"], {"event": "validate_request", "valid": valid}],
        }

    def execute_analysis_node(state):
        result = execute_analysis(state["decision"])
        return {
            **state,
            "result": result,
            "trace": [*state["trace"], {"event": "execute_analysis", "groups": sorted(result)}],
        }

    def validate_result_node(state):
        valid = validate_result(state["result"])
        return {
            **state,
            "result_valid": valid,
            "trace": [*state["trace"], {"event": "validate_result", "valid": valid}],
        }

    async def report_node(state):
        report_output = await report(client, state["question"], state["result"])
        valid = validate_report(report_output, state["result"])
        return {
            **state,
            "report": report_output,
            "outcome": "supported_report" if valid else "fallback",
            "termination": "criteria_met" if valid else "report_validation_failed",
            "model_calls": 2,
            "trace": [*state["trace"], {"event": "report", "valid": valid}],
        }

    def fallback_node(state):
        termination = (
            "invalid_analysis_request"
            if not state.get("request_valid", True)
            else "result_validation_failed"
        )
        return {**state, "outcome": "fallback", "termination": termination}

    graph = StateGraph(dict)
    graph.add_node("inspect_data", inspect_data_node)
    graph.add_node("select_analysis", select_analysis_node)
    graph.add_node("validate_request", validate_request_node)
    graph.add_node("execute_analysis", execute_analysis_node)
    graph.add_node("validate_result", validate_result_node)
    graph.add_node("report", report_node)
    graph.add_node("fallback", fallback_node)

    graph.add_edge(START, "inspect_data")
    graph.add_edge("inspect_data", "select_analysis")
    graph.add_edge("select_analysis", "validate_request")
    graph.add_conditional_edges(
        "validate_request",
        lambda state: "execute_analysis" if state["request_valid"] else "fallback",
    )
    graph.add_edge("execute_analysis", "validate_result")
    graph.add_conditional_edges(
        "validate_result",
        lambda state: "report" if state["result_valid"] else "fallback",
    )
    graph.add_edge("report", END)
    graph.add_edge("fallback", END)
    return graph.compile()


first = await build_graph().ainvoke({})
second = await build_graph().ainvoke({}) if MODEL_PROVIDER == "mock" else first
events = [item["event"] for item in first["trace"]]
evaluation = {
    "component": first.get("result") == expected["mean_reduction_kg"],
    "trajectory": events
    == [
        "inspect_data",
        "select_analysis",
        "validate_request",
        "execute_analysis",
        "validate_result",
        "report",
    ],
    "task": first["outcome"] == "supported_report",
    "safety": first["decision"].tool == "mean_reduction_by_intervention",
    "repeated_run": first["trace"] == second["trace"] and first["result"] == second["result"],
}
if MODEL_PROVIDER == "mock":
    assert all(evaluation.values()), {"evaluation": evaluation, "result": first}
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "termination": first["termination"],
    "resource_report": {"model_calls": first["model_calls"], "graph_nodes": 7},
}

## Limitation

This case study supports one allowlisted descriptive analysis. It does not permit arbitrary model-authored code and does not justify causal conclusions.
